## Final Project Submission

Please fill out:
* Student name: 
* Student pace: self paced / part time / full time
* Scheduled project review date/time: 
* Instructor name: 
* Blog post URL:


# Movie Studio Analysis

## Business Understanding

The goal of this project is to analyze movie industry data and provide actionable recommendations to a company planning to create a new movie studio.

We will analyze movie characteristics, ratings, genres, runtime, and box office performance to identify factors associated with successful movies.

## Data Understanding

The analysis will use movie industry datasets containing information about movies, ratings, genres, runtime, and box office performance.

The main datasets we will work with are:

- IMDb movie data stored in `im.db`
- Box Office Mojo data stored in `bom.movie_gross.csv`
- Additional movie datasets available in the `zippedData` folder

We will first inspect the available data, understand the columns and data types, and then determine which datasets are most useful for answering our business questions.

In [62]:
import pandas as pd
import sqlite3
import matplotlib.pyplot as plt

### Load the Box Office Mojo Dataset

We load the Box Office Mojo dataset into a pandas DataFrame so that we can inspect its structure and assess its data quality before cleaning.

In [66]:
# Load the Box Office Mojo movie gross dataset
bom = pd.read_csv("zippedData/bom.movie_gross.csv")

### Preview the Dataset

We display the first few records to understand the type of information contained in the dataset.

In [68]:
# Display the first five rows of the dataset
bom.head()

,title,studio,domestic_gross,foreign_gross,year
0,Toy Story 3,BV,415000000.0,652000000,2010
1,Alice in Wonderland (2010),BV,334200000.0,691300000,2010
2,Harry Potter and the Deathly Hallows Part 1,WB,296000000.0,664300000,2010
3,Inception,WB,292600000.0,535700000,2010
4,Shrek Forever After,P/DW,238700000.0,513900000,2010


### Dataset Dimensions

We check the number of rows and columns in the Box Office Mojo dataset to understand its size before carrying out further inspection and cleaning.

In [69]:
# Check the number of rows and columns
bom.shape

(3387, 5)

### Dataset Structure

We inspect the dataset structure to identify the column names, number of non-null observations, and data types. This helps us identify columns that may require cleaning.

In [70]:
# Display column names, non-null counts, and data types
bom.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3387 entries, 0 to 3386
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   title           3387 non-null   object 
 1   studio          3382 non-null   object 
 2   domestic_gross  3359 non-null   float64
 3   foreign_gross   2037 non-null   object 
 4   year            3387 non-null   int64  
dtypes: float64(1), int64(1), object(3)
memory usage: 132.4+ KB


### Missing Values

We check each column for missing values to identify incomplete records and determine which variables require attention during the cleaning stage.

In [71]:
# Count missing values in each column
bom.isna().sum()

title                0
studio               5
domestic_gross      28
foreign_gross     1350
year                 0
dtype: int64

### Duplicate Records

We check for completely duplicated rows to ensure that duplicate movie records do not cause movies or revenue to be counted more than once.

In [72]:
# Count completely duplicated rows
bom.duplicated().sum()

0

### Inspecting Box Office Revenue

The `domestic_gross` and `foreign_gross` columns contain box-office revenue. We inspect their values and data types before deciding how they should be cleaned and prepared for analysis.

In [73]:
# Inspect the data types of the revenue columns
bom[["domestic_gross", "foreign_gross"]].dtypes

domestic_gross    float64
foreign_gross      object
dtype: object

In [74]:
# Display sample revenue values
bom[["domestic_gross", "foreign_gross"]].head(10)

,domestic_gross,foreign_gross
0,415000000.0,652000000
1,334200000.0,691300000
2,296000000.0,664300000
3,292600000.0,535700000
4,238700000.0,513900000
5,300500000.0,398000000
6,312400000.0,311500000
7,200800000.0,391000000
8,251500000.0,291600000
9,217600000.0,277300000


## BOM Dataset Inspection Findings

The Box Office Mojo dataset contains **3,387 movie records and 5 variables**: `title`, `studio`, `domestic_gross`, `foreign_gross`, and `year`.

The inspection shows that `title` and `year` have complete records, while `studio`, `domestic_gross`, and `foreign_gross` contain missing values.

- `studio` has **5 missing values**.
- `domestic_gross` has **28 missing values**.
- `foreign_gross` has **1,350 missing values**.

The data types also reveal an important issue. `domestic_gross` is stored as a numeric (`float64`) variable, while `foreign_gross` is stored as an `object`. Therefore, `foreign_gross` will require further cleaning and conversion before it can be reliably used in numerical calculations.

The duplicate check returned **0 completely duplicated rows**, indicating that no identical records need to be removed from the dataset.

Overall, the inspection identifies missing values and an inconsistent data type in `foreign_gross` as the main data-quality issues to address during the cleaning stage.

## Data Preparation

Based on the inspection findings, the BOM dataset requires cleaning before it can be used for analysis.

The cleaning process will focus on:

1. Handling missing values in `studio`, `domestic_gross`, and `foreign_gross`.
2. Converting `foreign_gross` from an object to a numeric variable.
3. Checking the cleaned values for consistency.
4. Preparing the revenue variables for calculating total box-office performance.
5. Preserving valid movie records while avoiding unnecessary data loss.

## Data Cleaning

### Handling Missing Studio Values

The `studio` column identifies the movie's production or distribution studio.

Our inspection identified 5 records with missing studio values. Before deciding how to handle these records, we inspect them to determine whether the studio information can be recovered from the available data or whether the records should be excluded from studio-level analysis.

In [75]:
# Display movies with missing studio values
bom[bom["studio"].isna()]

,title,studio,domestic_gross,foreign_gross,year
210,Outside the Law (Hors-la-loi),NaN,96900.0,3300000,2010
555,Fireflies in the Garden,NaN,70600.0,3300000,2011
933,Keith Lemon: The Film,NaN,NaN,4000000,2012
1862,Plot for Peace,NaN,7100.0,NaN,2014
2825,Secret Superstar,NaN,NaN,122000000,2017


In [76]:
# Display the title, gross revenue, and year for movies with missing studios
bom.loc[
    bom["studio"].isna(),
    ["title", "domestic_gross", "foreign_gross", "year"]
]

,title,domestic_gross,foreign_gross,year
210,Outside the Law (Hors-la-loi),96900.0,3300000,2010
555,Fireflies in the Garden,70600.0,3300000,2011
933,Keith Lemon: The Film,NaN,4000000,2012
1862,Plot for Peace,7100.0,NaN,2014
2825,Secret Superstar,NaN,122000000,2017


### Handling Missing Studio Values

The inspection identified 5 movie records with missing values in the `studio` column.

The affected records contain valid movie information and should not be removed simply because the studio is unavailable. Since the available BOM data does not provide enough information to reliably determine the missing studios, we will replace the missing studio values with `"Unknown"`.

This approach preserves the movie records while clearly identifying that the studio information was unavailable. The `"Unknown"` category can be excluded from studio-level comparisons where appropriate.

In [77]:
# Replace missing studio values with "Unknown"
bom["studio"] = bom["studio"].fillna("Unknown")

### Verify Studio Cleaning

We verify that all missing values in the `studio` column have been handled successfully.

In [78]:
# Check the number of remaining missing studio values
bom["studio"].isna().sum()

0

In [81]:
# Confirm that five records were assigned to the "Unknown" category
bom["studio"].value_counts().get("Unknown", 0)

5

In [80]:
# Confirm that the number of rows remains unchanged after cleaning
bom.shape

(3387, 5)

### Studio Cleaning Result

The 5 missing values in the `studio` column were replaced with `"Unknown"`.

No movie records were removed during this process, so the dataset remains at 3,387 rows. Using an `"Unknown"` category preserves valid movie records while clearly indicating that the studio information was unavailable.

These `"Unknown"` records will be considered appropriately when performing studio-level analysis.

### Handling Missing Domestic Gross Values

The inspection identified 28 missing values in the `domestic_gross` column.

Domestic gross is an important measure of box-office performance, so we should not replace missing values with an arbitrary value such as zero without first investigating the affected records.

We will inspect the affected movies and determine an appropriate treatment while avoiding unnecessary loss of valid records.

In [82]:
# Display movies with missing domestic gross values
bom[bom["domestic_gross"].isna()]

,title,studio,domestic_gross,foreign_gross,year
230,It's a Wonderful Afterlife,UTV,NaN,1300000,2010
298,Celine: Through the Eyes of the World,Sony,NaN,119000,2010
302,White Lion,Scre.,NaN,99600,2010
306,Badmaash Company,Yash,NaN,64400,2010
327,Aashayein (Wishes),Relbig.,NaN,3800,2010
537,Force,FoxS,NaN,4800000,2011
713,Empire of Silver,NeoC,NaN,19000,2011
871,Solomon Kane,RTWC,NaN,19600000,2012
928,The Tall Man,Imag.,NaN,5200000,2012
933,Keith Lemon: The Film,Unknown,NaN,4000000,2012


In [83]:
# Display relevant information for movies with missing domestic gross
bom.loc[
    bom["domestic_gross"].isna(),
    ["title", "studio", "foreign_gross", "year"]
]


,title,studio,foreign_gross,year
230,It's a Wonderful Afterlife,UTV,1300000,2010
298,Celine: Through the Eyes of the World,Sony,119000,2010
302,White Lion,Scre.,99600,2010
306,Badmaash Company,Yash,64400,2010
327,Aashayein (Wishes),Relbig.,3800,2010
537,Force,FoxS,4800000,2011
713,Empire of Silver,NeoC,19000,2011
871,Solomon Kane,RTWC,19600000,2012
928,The Tall Man,Imag.,5200000,2012
933,Keith Lemon: The Film,Unknown,4000000,2012


In [84]:
# Count the remaining missing domestic gross values
bom["domestic_gross"].isna().sum()

28

### Handling Missing Domestic Gross Values

The inspection identified 28 missing values in the `domestic_gross` column.

The affected records are valid movie observations and several have reported `foreign_gross` values. Therefore, the missing domestic gross values should not automatically be interpreted as zero.

Because the actual domestic gross cannot be reliably determined from the BOM dataset, we will retain these records and leave the missing values as `NaN`. This avoids introducing inaccurate revenue values or unnecessarily removing valid movie records.

For analyses that specifically require domestic gross, pandas will exclude these missing observations from calculations by default.

In [85]:
# Confirm that the 28 domestic gross values are still missing
bom["domestic_gross"].isna().sum()

28

In [86]:
# Confirm that no movie records were removed
bom.shape

(3387, 5)

### Domestic Gross Cleaning Result

The 28 missing values in `domestic_gross` were retained as missing values because the actual domestic revenue cannot be reliably determined from the available data.

No rows were removed and no artificial revenue values were introduced. The dataset therefore remains at 3,387 records.

For analyses that require domestic gross, calculations will be based only on records with available domestic gross values.

### Handling Foreign Gross Values

The `foreign_gross` column contains foreign box-office revenue. During inspection, we identified 1,350 missing values and found that the column is stored as an `object` rather than a numeric data type.

Before performing numerical analysis, we need to convert the available foreign gross values to a numeric format. Missing values will remain as missing because there is not enough information in this dataset to reliably determine the actual foreign revenue.

In [87]:
# Display sample values from the foreign gross column
bom["foreign_gross"].head(20)

0     652000000
1     691300000
2     664300000
3     535700000
4     513900000
5     398000000
6     311500000
7     391000000
8     291600000
9     277300000
10    330000000
11    311300000
12    275400000
13    228000000
14    182500000
15    245600000
16    222400000
17    173500000
18    216400000
19    187900000
Name: foreign_gross, dtype: object

In [88]:
# Check the current data type
bom["foreign_gross"].dtype

dtype('O')

In [89]:
# Convert foreign gross values to numeric
# Invalid or missing values are converted to NaN
bom["foreign_gross"] = pd.to_numeric(
    bom["foreign_gross"],
    errors="coerce"
)

### Verify Foreign Gross Conversion

After conversion, we check the data type and missing-value count to confirm that the revenue variable is ready for numerical analysis.

In [90]:
# Check the new data type
bom["foreign_gross"].dtype

dtype('float64')

In [91]:
# Check the number of missing foreign gross values
bom["foreign_gross"].isna().sum()

1355

In [92]:
# Display summary statistics for foreign gross
bom["foreign_gross"].describe()

count    2.032000e+03
mean     7.505704e+07
std      1.375294e+08
min      6.000000e+02
25%      3.775000e+06
50%      1.890000e+07
75%      7.505000e+07
max      9.605000e+08
Name: foreign_gross, dtype: float64

### Identifying Non-Numeric Foreign Gross Values

After converting `foreign_gross` to a numeric data type, we investigate the values that could not be converted. This helps us understand why the number of missing values increased after conversion and ensures that the cleaning process is properly documented.

In [93]:
# Reload the original foreign gross values for comparison
bom_original = pd.read_csv("zippedData/bom.movie_gross.csv")

# Identify values that were present but could not be converted to numeric
original_foreign = bom_original["foreign_gross"]

non_numeric = original_foreign[
    original_foreign.notna() &
    pd.to_numeric(original_foreign, errors="coerce").isna()
]

non_numeric

1872    1,131.6
1873    1,019.4
1874    1,163.0
2760    1,010.0
3079    1,369.5
Name: foreign_gross, dtype: object

### Cleaning Non-Numeric Foreign Gross Values

The investigation identified 5 `foreign_gross` values that could not initially be converted to numeric values.

These values contain commas as thousands separators, for example `1,131.6`. The values represent valid numerical amounts, so they should not be treated as missing.

We will remove the commas and convert these values to numeric format.

In [98]:
# Restore the original foreign gross values
bom["foreign_gross"] = bom_original["foreign_gross"]

# Remove thousands separators and convert the column to numeric
bom["foreign_gross"] = (
    bom["foreign_gross"]
    .astype("string")
    .str.replace(",", "", regex=False)
    .pipe(pd.to_numeric, errors="coerce")
)

In [99]:
# Confirm that foreign gross is now numeric
bom["foreign_gross"].dtype

dtype('float64')

In [100]:
# Check the number of missing foreign gross values after cleaning
bom["foreign_gross"].isna().sum()

1350

In [101]:
# Confirm that the previously non-numeric values were successfully converted
bom.loc[
    [1872, 1873, 1874, 2760, 3079],
    "foreign_gross"
]

1872    1131.6
1873    1019.4
1874    1163.0
2760    1010.0
3079    1369.5
Name: foreign_gross, dtype: float64

### Foreign Gross Cleaning Result

The `foreign_gross` column was successfully converted from an object data type to a numeric (`float64`) data type.

During the cleaning process, 5 values containing commas as thousands separators were identified. These values were cleaned and successfully converted to numeric values rather than being treated as missing.

After cleaning, the dataset contains 1,350 missing `foreign_gross` values, which matches the original missing-value count.

The remaining missing values were retained as `NaN` because their actual foreign box-office revenue cannot be reliably determined from the available dataset.

## Final BOM Cleaning Check

After completing the cleaning process, we verify the structure, missing values, data types, and duplicate records of the BOM dataset.

The cleaning process preserved valid movie records, standardized the `studio` column, converted `foreign_gross` to a numeric format, and retained unavailable revenue values as missing rather than introducing artificial values.

In [102]:
# Check the final shape of the cleaned BOM dataset
bom.shape

(3387, 5)

In [103]:
# Check remaining missing values
bom.isna().sum()

title                0
studio               0
domestic_gross      28
foreign_gross     1350
year                 0
dtype: int64

In [104]:
# Check final data types
bom.dtypes

title              object
studio             object
domestic_gross    float64
foreign_gross     float64
year                int64
dtype: object

In [105]:
# Confirm that no duplicate rows were introduced
bom.duplicated().sum()

0

## Merging the Datasets

The cleaned BOM movie gross dataset will be combined with IMDb movie information to support the project analysis.

The IMDb `movie_basics` and `movie_ratings` tables will first be merged using `movie_id`. The resulting dataset will then be joined with the BOM movie gross data using movie title.

This will allow us to analyze movie characteristics, ratings, and box-office performance together.

## Load IMDb Movie Ratings

The IMDb movie ratings table contains ratings information for the movies in the database. We load this table from the SQLite database so that it can later be combined with the movie information and BOM box-office data.

In [106]:
# Load the IMDb movie ratings table from the SQLite database
movie_ratings = pd.read_sql_query(
    "SELECT * FROM movie_ratings",
    conn
)

In [107]:
# Display the first five rows of the movie ratings dataset
movie_ratings.head()

,movie_id,averagerating,numvotes
0,tt10356526,8.3,31
1,tt10384606,8.9,559
2,tt1042974,6.4,20
3,tt1043726,4.2,50352
4,tt1060240,6.5,21


In [108]:
# Check the structure of the movie ratings dataset
movie_ratings.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 73856 entries, 0 to 73855
Data columns (total 3 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   movie_id       73856 non-null  object 
 1   averagerating  73856 non-null  float64
 2   numvotes       73856 non-null  int64  
dtypes: float64(1), int64(1), object(1)
memory usage: 1.7+ MB


In [109]:
# Check the dimensions of the movie ratings dataset
movie_ratings.shape

(73856, 3)

### Load IMDb Movie Basics

The IMDb movie basics table contains core information about each movie, including its unique `movie_id`. This identifier will be used to connect the movie basics information with the movie ratings data.

In [110]:
# Display the first five rows of the IMDb movie basics dataset
movie_basics.head()

,movie_id,primary_title,original_title,start_year,runtime_minutes,genres
0,tt0063540,Sunghursh,Sunghursh,2013,175.0,"Action,Crime,Drama"
1,tt0066787,One Day Before the Rainy Season,Ashad Ka Ek Din,2019,114.0,"Biography,Drama"
2,tt0069049,The Other Side of the Wind,The Other Side of the Wind,2018,122.0,Drama
3,tt0069204,Sabse Bada Sukh,Sabse Bada Sukh,2018,NaN,"Comedy,Drama"
4,tt0100275,The Wandering Soap Opera,La Telenovela Errante,2017,80.0,"Comedy,Drama,Fantasy"


In [111]:
# Check the structure of the IMDb movie basics dataset
movie_basics.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 146144 entries, 0 to 146143
Data columns (total 6 columns):
 #   Column           Non-Null Count   Dtype  
---  ------           --------------   -----  
 0   movie_id         146144 non-null  object 
 1   primary_title    146144 non-null  object 
 2   original_title   146123 non-null  object 
 3   start_year       146144 non-null  int64  
 4   runtime_minutes  114405 non-null  float64
 5   genres           140736 non-null  object 
dtypes: float64(1), int64(1), object(4)
memory usage: 6.7+ MB


In [112]:
# Check the dimensions of the IMDb movie basics dataset
movie_basics.shape

(146144, 6)

## Merge IMDb Movie Basics and Ratings

The `movie_basics` and `movie_ratings` datasets share the `movie_id` identifier. We use this common key to combine movie characteristics with their ratings and vote counts.

An inner merge is used so that the resulting dataset contains movies that have information available in both IMDb tables.

In [113]:
# Merge IMDb movie basics with movie ratings using movie_id
imdb_merged = pd.merge(
    movie_basics,
    movie_ratings,
    on="movie_id",
    how="inner"
)

In [114]:
# Display the first five rows of the merged IMDb dataset
imdb_merged.head()

,movie_id,primary_title,original_title,start_year,runtime_minutes,genres,averagerating,numvotes
0,tt0063540,Sunghursh,Sunghursh,2013,175.0,"Action,Crime,Drama",7.0,77
1,tt0066787,One Day Before the Rainy Season,Ashad Ka Ek Din,2019,114.0,"Biography,Drama",7.2,43
2,tt0069049,The Other Side of the Wind,The Other Side of the Wind,2018,122.0,Drama,6.9,4517
3,tt0069204,Sabse Bada Sukh,Sabse Bada Sukh,2018,NaN,"Comedy,Drama",6.1,13
4,tt0100275,The Wandering Soap Opera,La Telenovela Errante,2017,80.0,"Comedy,Drama,Fantasy",6.5,119


In [115]:
# Check the structure of the merged IMDb dataset
imdb_merged.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 73856 entries, 0 to 73855
Data columns (total 8 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   movie_id         73856 non-null  object 
 1   primary_title    73856 non-null  object 
 2   original_title   73856 non-null  object 
 3   start_year       73856 non-null  int64  
 4   runtime_minutes  66236 non-null  float64
 5   genres           73052 non-null  object 
 6   averagerating    73856 non-null  float64
 7   numvotes         73856 non-null  int64  
dtypes: float64(2), int64(2), object(4)
memory usage: 5.1+ MB


In [116]:
# Check the dimensions of the merged IMDb dataset
imdb_merged.shape

(73856, 8)

In [117]:
# Check whether the merged IMDb dataset contains duplicate movie IDs
imdb_merged["movie_id"].duplicated().sum()

0

### IMDb Merge Result

The `movie_basics` and `movie_ratings` datasets were successfully merged using `movie_id`.

The resulting dataset contains 73,856 movie records and 8 columns. The duplicate check returned zero duplicate `movie_id` values, confirming that each movie appears only once in the merged IMDb dataset.

The merged dataset now combines movie characteristics, ratings, and vote counts and is ready to be joined with the cleaned BOM dataset.

In [118]:
# Check the title columns that will be used for the merge
imdb_merged["primary_title"].head()

0                          Sunghursh
1    One Day Before the Rainy Season
2         The Other Side of the Wind
3                    Sabse Bada Sukh
4           The Wandering Soap Opera
Name: primary_title, dtype: object

In [119]:
# Check the BOM movie titles
bom["title"].head()

0                                    Toy Story 3
1                     Alice in Wonderland (2010)
2    Harry Potter and the Deathly Hallows Part 1
3                                      Inception
4                            Shrek Forever After
Name: title, dtype: object

In [120]:
# Check the number of unique titles in each dataset
print("IMDb unique titles:", imdb_merged["primary_title"].nunique())
print("BOM unique titles:", bom["title"].nunique())

IMDb unique titles: 69993
BOM unique titles: 3386


### Checking Title Compatibility

The IMDb and BOM datasets do not contain the same number of unique movie titles. Before merging the datasets, we therefore check how many BOM titles have an exact match in the IMDb `primary_title` column.

This helps us understand how much of the BOM box-office data can be connected to the IMDb movie information using title.

In [121]:
# Identify BOM titles that have an exact match in IMDb
bom["title"].isin(imdb_merged["primary_title"]).sum()

2599

In [122]:
# Count BOM titles that do not have an exact IMDb title match
(~bom["title"].isin(imdb_merged["primary_title"])).sum()

788

In [123]:
# Calculate the percentage of BOM titles that have an exact IMDb match
match_rate = (
    bom["title"].isin(imdb_merged["primary_title"]).mean() * 100
)

match_rate

76.73457336876291

In [124]:
# Display a sample of BOM titles that do not have an exact IMDb match
bom.loc[
    ~bom["title"].isin(imdb_merged["primary_title"]),
    "title"
].head(20)

1                            Alice in Wonderland (2010)
2           Harry Potter and the Deathly Hallows Part 1
10                           Clash of the Titans (2010)
13                                          Tron Legacy
28                                         Knight & Day
31    Percy Jackson & The Olympians: The Lightning T...
41                                          Jackass 3-D
44                                          Step Up 3-D
51                                               Saw 3D
55                     A Nightmare on Elm Street (2010)
62                                 Love and Other Drugs
63                       Aftershock (Tangshan Dadizhen)
74                                               Oceans
78                                     The Back-Up Plan
79           If You Are the One 2 (Fei Cheng Wu Rao II)
82                                            Hubble 3D
84                                  Tales from Earthsea
85                                        Legion

### Investigating Unmatched Movie Titles

The exact title comparison matched 2,599 of the 3,387 BOM movie records, giving an exact match rate of approximately 76.7%.

The remaining 788 titles do not have an exact match in the IMDb `primary_title` column. Before merging, we investigate these unmatched titles to determine whether the differences are caused by formatting, alternate titles, or genuinely missing IMDb matches.

We will not force unmatched records into the merge because incorrect title matching could associate a movie with the wrong IMDb rating or movie information.

In [125]:
# Display examples of unmatched BOM movie titles
unmatched_bom = bom[
    ~bom["title"].isin(imdb_merged["primary_title"])
]

unmatched_bom[["title", "studio", "year"]].head(20)

,title,studio,year
1,Alice in Wonderland (2010),BV,2010
2,Harry Potter and the Deathly Hallows Part 1,WB,2010
10,Clash of the Titans (2010),WB,2010
13,Tron Legacy,BV,2010
28,Knight & Day,Fox,2010
31,Percy Jackson & The Olympians: The Lightning T...,Fox,2010
41,Jackass 3-D,Par.,2010
44,Step Up 3-D,BV,2010
51,Saw 3D,LGF,2010
55,A Nightmare on Elm Street (2010),WB (NL),2010


In [126]:
# Compare the year distribution of unmatched BOM movies
unmatched_bom["year"].value_counts().sort_index()

2010    133
2011     93
2012     78
2013     72
2014     90
2015    100
2016     85
2017     62
2018     75
Name: year, dtype: int64

In [127]:
# Identify duplicate movie titles in the BOM dataset
bom[bom["title"].duplicated(keep=False)].sort_values("title")

,title,studio,domestic_gross,foreign_gross,year
317,Bluebeard,Strand,33500.0,5200.0,2010
3045,Bluebeard,WGUSA,43100.0,NaN,2017


### Duplicate Title Investigation

The BOM dataset contains one duplicated movie title: `Bluebeard`.

The two records have different studios, release years, and box-office values. Therefore, the duplicate title does not necessarily represent a duplicate movie record.

Both records will be retained because the title alone is not sufficient evidence that they are duplicates.

### Standardizing Movie Titles for Matching

The BOM and IMDb datasets use movie titles as the connection field, but formatting differences can prevent exact matches.

We will standardize capitalization and remove leading or trailing whitespace before comparing the titles. This improves consistency without changing the underlying movie names.

In [128]:
# Create standardized versions of the movie titles for matching
bom["title_clean"] = bom["title"].str.strip().str.lower()

imdb_merged["title_clean"] = (
    imdb_merged["primary_title"]
    .str.strip()
    .str.lower()
)

In [129]:
# Check the number of BOM titles matching IMDb after title standardization
bom["title_clean"].isin(imdb_merged["title_clean"]).sum()

2695

In [130]:
# Calculate the standardized title match rate
match_rate_clean = (
    bom["title_clean"].isin(imdb_merged["title_clean"]).mean() * 100
)

match_rate_clean

79.56894006495425

### Investigating Remaining Unmatched Titles

After standardizing capitalization and removing leading and trailing whitespace, the number of BOM titles matching IMDb increased from 2,599 to 2,695.

This improved the exact title match rate from 76.73% to 79.57%.

There are still 692 BOM records without an exact standardized title match. These records will be investigated before the merge to avoid incorrectly assigning IMDb information to the wrong movie.

In [131]:
# Identify BOM records that still do not have a standardized IMDb title match
unmatched_bom = bom[
    ~bom["title_clean"].isin(imdb_merged["title_clean"])
]

unmatched_bom[["title", "studio", "year"]].head(30)

,title,studio,year
1,Alice in Wonderland (2010),BV,2010
2,Harry Potter and the Deathly Hallows Part 1,WB,2010
10,Clash of the Titans (2010),WB,2010
13,Tron Legacy,BV,2010
28,Knight & Day,Fox,2010
41,Jackass 3-D,Par.,2010
44,Step Up 3-D,BV,2010
51,Saw 3D,LGF,2010
55,A Nightmare on Elm Street (2010),WB (NL),2010
62,Love and Other Drugs,Fox,2010


In [132]:
# Count the remaining unmatched BOM records
unmatched_bom.shape

(692, 6)

### Merge BOM with IMDb Data

After standardizing movie titles, 2,695 BOM records have matching titles in the IMDb dataset.

Because BOM is the primary dataset for this analysis, we use a left merge to retain all BOM movie records. Movies without a matching IMDb record will have missing IMDb information rather than being incorrectly matched.

This approach preserves the complete BOM box-office dataset while adding available IMDb ratings and movie characteristics.

In [133]:
# Merge the cleaned BOM data with the IMDb dataset
bom_imdb = pd.merge(
    bom,
    imdb_merged,
    left_on="title_clean",
    right_on="title_clean",
    how="left"
)

In [134]:
# Check the shape of the merged dataset
bom_imdb.shape

(3825, 14)

In [135]:
# Inspect the merged dataset
bom_imdb.head()

,title,studio,domestic_gross,foreign_gross,year,title_clean,movie_id,primary_title,original_title,start_year,runtime_minutes,genres,averagerating,numvotes
0,Toy Story 3,BV,415000000.0,652000000.0,2010,toy story 3,tt0435761,Toy Story 3,Toy Story 3,2010.0,103.0,"Adventure,Animation,Comedy",8.3,682218.0
1,Alice in Wonderland (2010),BV,334200000.0,691300000.0,2010,alice in wonderland (2010),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Harry Potter and the Deathly Hallows Part 1,WB,296000000.0,664300000.0,2010,harry potter and the deathly hallows part 1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Inception,WB,292600000.0,535700000.0,2010,inception,tt1375666,Inception,Inception,2010.0,148.0,"Action,Adventure,Sci-Fi",8.8,1841066.0
4,Shrek Forever After,P/DW,238700000.0,513900000.0,2010,shrek forever after,tt0892791,Shrek Forever After,Shrek Forever After,2010.0,93.0,"Adventure,Animation,Comedy",6.3,167532.0


In [136]:
# Check how many BOM records received an IMDb rating
bom_imdb["averagerating"].notna().sum()

3133

In [137]:
# Check how many BOM records did not find an IMDb match
bom_imdb["averagerating"].isna().sum()

692

### Validating the BOM-IMDb Merge

The initial left merge produced 3,825 rows compared with the original 3,387 BOM records.

Because a left merge should preserve the BOM records, the increase indicates that some standardized movie titles correspond to multiple IMDb records. We will investigate duplicate `title_clean` values in the IMDb dataset before finalizing the merge.

This check is necessary to prevent one BOM movie from being associated with multiple IMDb movies.

In [138]:
# Check for duplicate standardized titles in IMDb
imdb_merged["title_clean"].duplicated().sum()

3915

In [139]:
# Display IMDb titles that appear more than once
duplicate_imdb_titles = imdb_merged[
    imdb_merged["title_clean"].duplicated(keep=False)
].sort_values("title_clean")

duplicate_imdb_titles[
    ["movie_id", "primary_title", "start_year", "title_clean"]
].head(30)

,movie_id,primary_title,start_year,title_clean
12250,tt1859618,1,2019,1
25949,tt2518788,1,2013,1
52873,tt5282238,10,2015,10
13822,tt1941430,10,2011,10
12486,tt1869226,100% Love,2011,100% love
26727,tt2579680,100% Love,2012,100% love
61951,tt6474850,11,2017,11
33149,tt3187586,11,2014,11
12111,tt1855110,180,2011,180
14576,tt1979163,180,2011,180


In [140]:
# Count how many unique standardized titles are duplicated
imdb_merged["title_clean"].value_counts().loc[
    lambda x: x > 1
].head(20)

the return      11
broken          10
homecoming       9
lucky            9
together         9
paradise         8
home             8
brothers         8
rage             8
dead end         8
the gift         8
delirium         8
angel            7
alone            7
solo             7
the end          7
split            6
karma            6
gold             6
the stranger     6
Name: title_clean, dtype: int64

### Refining the BOM-IMDb Merge Key

The initial title-only merge created additional rows because some IMDb movies share the same title.

To reduce incorrect matches, we will use both the standardized movie title and release year as the merge keys. BOM provides `title` and `year`, while IMDb provides `primary_title` and `start_year`.

Using title and year together provides a more specific connection between the two datasets and reduces the risk of assigning the wrong IMDb rating to a BOM movie.

In [141]:
# Create standardized year and title fields for the IMDb merge
imdb_merged["year"] = imdb_merged["start_year"].astype("Int64")

In [142]:
# Merge BOM with IMDb using standardized title and release year
bom_imdb_year = pd.merge(
    bom,
    imdb_merged,
    left_on=["title_clean", "year"],
    right_on=["title_clean", "year"],
    how="left"
)

In [143]:
# Check the shape of the refined merge
bom_imdb_year.shape

(3413, 14)

In [144]:
# Check how many BOM records received an IMDb rating
bom_imdb_year["averagerating"].notna().sum()

1921

In [145]:
# Check how many BOM records remain without an IMDb rating
bom_imdb_year["averagerating"].isna().sum()

1492

In [146]:
# Check whether the refined merge still created duplicate BOM records
bom_imdb_year.shape[0] - bom.shape[0]

26

### Investigating Duplicate Title-Year Matches

The title-and-year merge reduced the number of duplicated records substantially, but the merged dataset still contains 26 additional rows compared with the original BOM dataset.

This indicates that some IMDb records share the same standardized title and release year. These duplicate title-year combinations must be investigated before finalizing the merge to avoid assigning multiple IMDb records to a single BOM movie.

In [ ]:
# Identify duplicate title-year combinations in IMDb
duplicate_title_years = (
    imdb_merged
    .groupby(["title_clean", "year"])
    .size()
    .reset_index(name="count")
)

duplicate_title_years = duplicate_title_years[
    duplicate_title_years["count"] > 1
]

duplicate_title_years.sort_values(
    "count", ascending=False
).head(20)



,title_clean,year,count
54422,split,2016,6
63538,the return,2015,4
11887,cargo,2017,4
64410,the stranger,2014,4
11967,carte blanche,2011,3
26120,henry,2010,3
5446,anna,2015,3
27724,i am,2010,3
53737,solo,2013,3
6733,ava,2017,3


### Resolving Duplicate IMDb Title-Year Records

Some IMDb movies share the same standardized title and release year but have different `movie_id` values and ratings.

To ensure that each BOM movie matches at most one IMDb record, we retain the IMDb record with the highest number of votes for each `title_clean` and `year` combination.

Using the record with the highest number of votes provides a consistent way to select the most widely supported IMDb rating while preventing duplicate BOM records during the merge.

In [151]:
# Sort IMDb records by number of votes
# The most-voted record will appear first for each title-year combination
imdb_sorted = imdb_merged.sort_values(
    "numvotes",
    ascending=False
)

In [152]:
# Keep one IMDb record for each standardized title and year
# We keep the record with the highest number of votes
imdb_unique = imdb_sorted.drop_duplicates(
    subset=["title_clean", "year"],
    keep="first"
)

In [153]:
# Confirm that there are no duplicate title-year combinations
imdb_unique.duplicated(
    subset=["title_clean", "year"]
).sum()

0

In [154]:
# Merge BOM with the unique IMDb records
bom_imdb_final = pd.merge(
    bom,
    imdb_unique,
    on=["title_clean", "year"],
    how="left"
)

In [155]:
# Check the final merged dataset shape
bom_imdb_final.shape

(3387, 14)

In [156]:
# Confirm that the merge did not create extra BOM records
bom_imdb_final.shape[0] - bom.shape[0]

0

### Final BOM-IMDb Merge Verification

The final merge was performed using standardized movie title and release year.

Duplicate IMDb title-year combinations were resolved by retaining the record with the highest number of votes. This ensured that each BOM movie matched at most one IMDb record.

The final merged dataset contains 3,387 records, the same number as the original BOM dataset, confirming that no BOM records were duplicated or lost during the merge.

In [157]:
# Inspect the final merged dataset
bom_imdb_final.head()

,title,studio,domestic_gross,foreign_gross,year,title_clean,movie_id,primary_title,original_title,start_year,runtime_minutes,genres,averagerating,numvotes
0,Toy Story 3,BV,415000000.0,652000000.0,2010,toy story 3,tt0435761,Toy Story 3,Toy Story 3,2010.0,103.0,"Adventure,Animation,Comedy",8.3,682218.0
1,Alice in Wonderland (2010),BV,334200000.0,691300000.0,2010,alice in wonderland (2010),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Harry Potter and the Deathly Hallows Part 1,WB,296000000.0,664300000.0,2010,harry potter and the deathly hallows part 1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Inception,WB,292600000.0,535700000.0,2010,inception,tt1375666,Inception,Inception,2010.0,148.0,"Action,Adventure,Sci-Fi",8.8,1841066.0
4,Shrek Forever After,P/DW,238700000.0,513900000.0,2010,shrek forever after,tt0892791,Shrek Forever After,Shrek Forever After,2010.0,93.0,"Adventure,Animation,Comedy",6.3,167532.0


In [158]:
# Check the final structure
bom_imdb_final.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 3387 entries, 0 to 3386
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   title            3387 non-null   object 
 1   studio           3387 non-null   object 
 2   domestic_gross   3359 non-null   float64
 3   foreign_gross    2037 non-null   float64
 4   year             3387 non-null   int64  
 5   title_clean      3387 non-null   object 
 6   movie_id         1895 non-null   object 
 7   primary_title    1895 non-null   object 
 8   original_title   1895 non-null   object 
 9   start_year       1895 non-null   float64
 10  runtime_minutes  1895 non-null   float64
 11  genres           1895 non-null   object 
 12  averagerating    1895 non-null   float64
 13  numvotes         1895 non-null   float64
dtypes: float64(6), int64(1), object(7)
memory usage: 396.9+ KB


In [159]:
# Check how many movies have IMDb ratings
bom_imdb_final["averagerating"].notna().sum()

1895

In [160]:
# Check how many movies do not have IMDb ratings
bom_imdb_final["averagerating"].isna().sum()

1492

## Studio-Level Box Office Aggregation

After cleaning the BOM movie gross data and merging it with the IMDb movie information, the next step is to evaluate box-office performance at the studio level.

Studios release different numbers of movies, and the success of individual movies can vary considerably. Looking at the data at the studio level allows us to compare studios based on both the number of movies they released and the box-office revenue generated by those movies.

We will group the merged dataset by `studio` and calculate the following measures:

- **Movie count:** the number of movies associated with each studio.
- **Total domestic gross:** the combined domestic box-office revenue generated by the studio's movies.
- **Total foreign gross:** the combined foreign box-office revenue generated by the studio's movies.
- **Average domestic gross:** the average domestic box-office revenue per movie for each studio.
- **Average foreign gross:** the average foreign box-office revenue per movie for each studio.

These measures will help us understand differences in studio performance. In particular, average gross will allow us to compare the typical box-office performance of movies from different studios, while total gross will show the overall revenue generated by each studio.

The resulting studio-level table will be used in the later analysis to identify patterns in studio performance and support evidence-based business recommendations.

In [161]:
# Aggregate box-office performance by studio
studio_summary = (
    bom_imdb_final
    .groupby("studio")
    .agg(
        movie_count=("title", "count"),
        total_domestic_gross=("domestic_gross", "sum"),
        total_foreign_gross=("foreign_gross", "sum"),
        average_domestic_gross=("domestic_gross", "mean"),
        average_foreign_gross=("foreign_gross", "mean")
    )
    .reset_index()
)

studio_summary.head()

,studio,movie_count,total_domestic_gross,total_foreign_gross,average_domestic_gross,average_foreign_gross
0,3D,1,6100000.0,9900000.0,6.100000e+06,9.900000e+06
1,A23,2,164200.0,0.0,8.210000e+04,NaN
2,A24,49,324194200.0,238462200.0,6.616208e+06,1.255064e+07
3,ADC,2,248200.0,0.0,1.241000e+05,NaN
4,AF,6,2142900.0,3500000.0,3.571500e+05,1.750000e+06


## Comparing Studios by Average Domestic Gross

To identify studios whose movies tend to perform strongly in the domestic box office market, we will rank the studios according to their average domestic gross.

Average domestic gross is useful because studios release different numbers of movies. A studio with many movies may have a high total gross simply because it released more movies, while the average domestic gross gives us an indication of the typical performance of a movie released by that studio.

We will therefore sort the studio-level summary from the highest to the lowest average domestic gross and examine the top-performing studios.

In [162]:
# Rank studios by average domestic gross
top_studios = studio_summary.sort_values(
    "average_domestic_gross",
    ascending=False
)

top_studios.head(10)

,studio,movie_count,total_domestic_gross,total_foreign_gross,average_domestic_gross,average_foreign_gross
36,BV,106,1.841903e+10,2.579385e+10,1.737644e+08,2.480178e+08
171,P/DW,10,1.682900e+09,3.393600e+09,1.682900e+08,3.393600e+08
248,WB (NL),45,3.995700e+09,6.339000e+09,8.879333e+07,1.440682e+08
238,Uni.,147,1.290239e+10,1.685477e+10,8.777138e+07,1.170470e+08
247,WB,140,1.216805e+10,1.866790e+10,8.691461e+07,1.435993e+08
93,Fox,136,1.094950e+10,2.005587e+10,8.051103e+07,1.496706e+08
215,Sony,110,8.459683e+09,1.394535e+10,7.761177e+07,1.315600e+08
185,Par.,101,7.685871e+09,1.186338e+10,7.609773e+07,1.262062e+08
140,MGM,3,2.000000e+08,1.126000e+08,6.666667e+07,5.630000e+07
219,Sum.,15,9.318710e+08,1.354900e+09,6.212473e+07,9.677857e+07


### Interpretation of Studio-Level Aggregation

The aggregation shows substantial differences in box-office performance across studios.

BV has the highest total domestic and foreign gross among the studios shown, while Universal (`Uni.`) and Warner Bros. (`WB`) have relatively high movie counts and substantial total revenue.

Some studios have high average gross despite releasing relatively few movies. For example, P/DW has only 10 movies but a high average domestic and foreign gross. This demonstrates why average performance should be considered alongside the number of movies released.

Studios with very small movie counts, such as MGM with only 3 movies, should be interpreted cautiously because a small number of releases can make the average gross highly sensitive to individual successful movies.

The studio-level aggregation therefore provides both scale and performance measures that will be useful for the next analysis of average gross per studio.